# 下载 Llava-1.5-7b 模型并进行推理

按照 [Hugging Face transformers 库文档](https://huggingface.co/docs/transformers/main/en/model_doc/llava)，演示如何通过 transformers 库使用 Llava-1.5-7b 进行推理。

## 加载依赖

Hugging Face 的 transformers 库已经对 Llava 模型做了适配。我们直接使用实现好的模块。

In [ ]:
import requests
from PIL import Image

import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, LlavaProcessor

## 下载模型

在 python 环境中运行下面的代码：安装库（已经安装就不用再装一遍）、切换下载源为镜像源、下载模型到本地指定路径。

```shell
pip install -U huggingface_hub
export HF_ENDPOINT=https://hf-mirror.com
huggingface-cli download --resume-download llava-hf/llava-1.5-7b-hf --local-dir llava-1.5-7b-hf --local-dir-use-symlinks False
```

### 命令逐段解析

```bash
huggingface-cli download --resume-download llava-hf/llava-1.5-7b-hf --local-dir llava-1.5-7b-hf --local-dir-use-symlinks False
```

*   **`huggingface-cli`**：这是 Hugging Face 官方的命令行工具。
*   **`download`**：下载指令。
*   **`--resume-download`**：**断点续传**。如果下载中途中断，再次运行命令时会从断开的地方继续，不用从头开始。
*   **`llava-hf/llava-1.5-7b-hf`**：这是模型在 Hugging Face Hub 上的**仓库 ID**（`用户名/模型名`），指定了要下载哪个模型。
*   **`--local-dir llava-1.5-7b-hf`**：**指定本地保存目录**。它告诉工具，不要用默认的隐藏缓存路径，而是将模型文件直接放进 `llava-1.5-7b-hf` 这个文件夹里。
*   **`--local-dir-use-symlinks False`**：**禁用符号链接**。这个参数设为 `False` 时，工具会**复制真实的模型文件**到目标目录，而不是创建指向默认缓存（`~/.cache/huggingface/hub`）的快捷方式（符号链接）。这样做的好处是得到的 `llava-1.5-7b-hf` 文件夹是**完全独立**的，可以随意移动、复制或打包，不用担心破坏了与隐藏缓存的链接关系。

### 模型会下载到哪个路径？

模型会下载到**当前终端所在的工作目录**下的 `llava-1.5-7b-hf` 文件夹中。

因为提供的 `--local-dir` 参数值 `llava-1.5-7b-hf` 是一个**相对路径**。例如，如果你在 `/home/user/projects` 目录下运行此命令，模型最终的存放路径就是：
`/home/user/projects/llava-1.5-7b-hf/`

如果你希望保存到其他固定位置，可以将 `--local-dir` 的值改为一个**绝对路径**，例如：`--local-dir /data/models/llava-1.5-7b-hf`。

### 常见问题

如果命令行提示找不到 huggingface-cli 命令，是环境变量没设置好导致的。自行找相关的教程，把对应 python 环境下的 Scripts 文件夹添加进系统环境变量。完成后开一个新的 shell 窗口即可。

## 加载模型

把模型加载到内存，然后加载到显卡。
以 `bfloat16` 精度加载 7B 模型约需 **14–15 GB 显存**。

In [ ]:
model_name_or_path = "llava-1.5-7b-hf"  # 指向刚才下载好的文件夹
device = "cuda:0"   # 不做检查，默认有 NVIDIA 显卡

model = LlavaForConditionalGeneration.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).to(device)

## 加载并测试输入处理模块

对于 LLaVA 模型，在训练时给定了严格的格式，所以推理时我们也需要把自己的输入规范到这个格式，便于模型识别。在我们下载的文件中包含了相关的处理模块，只需要加载即可。

In [ ]:
processor = LlavaProcessor.from_pretrained(model_name_or_path)
# 等价于：AutoProcessor

### 演示输入文本处理

假设我们需要问 LLaVA 图片中都有些什么，即用户输入为图片（另外上传，后面会演示）和一句话。按照 [Hugging Face transformers 库文档](https://huggingface.co/docs/transformers/main/en/model_doc/llava) 我们需要这样组织字段，然后传给 processor 的 apply_chat_template 进行处理。

这里需要注意的是，在 LLaVA 的原始代码库（`haotian-liu/LLaVA`）中，系统提示被定义在对话模板（`conv_templates`）里，是 `Conversation` 类的一个核心字段。

以 LLaVA-1.5 主要使用的 `vicuna_v1` 模板为例，其 `system` 字段被硬编码为：
> **"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the human's questions."**
> 

这个系统提示会在生成最终 Prompt 时，被 `get_prompt()` 方法自动添加到所有用户消息的最前面。

这里不指定 system 字段的内容，是因为需要模仿 transformers 库文档给出的示例。HF 的 `transformers` 库在实现 LLaVA 的 `chat_template` 时，为了**让演示代码更简洁**，选择性地移除了这个系统提示。

HF 的维护者在其代码仓库的 Pull Request 中明确解释道：
> “从原始 Llava 仓库的推理代码来看，正确的模板是……**技术上我们应该在这里添加系统提示，但我认为为了保持演示的简洁性，可以将其排除。**”

因此，HF 的 `apply_chat_template` 默认行为（直接从 `USER:` 开始）是一个**功能性简化**，并非对原始模型行为的严格复现。但实现中这不会导致明显的问题，体现了 LLM 对输入的鲁棒性。

In [ ]:
prompt_text = "What are these?"
messages = [
    {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_text},
            ]},
]
# 不做 tokenize，便于打印展示
prompt_format = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print(prompt_format)

### 输入图片处理

我们从本地将 `000000039769.jpg` 给模型，作为图片输入。

`apply_chat_template` 只负责生成带有 `<image>` 占位符的文本提示。图像像素必须通过 `processor(images=...)` 或 `apply_chat_template(..., images=...)` 显式传入，否则模型将看不到图片。

#### Processor 做了什么？

`LlavaProcessor` 将文本处理和图片处理整合在一个接口中。

在本示例里：

- `text=prompt`：提供已经构造好的提示词。
- `images=raw_image`：提供真实的图片数据。
- `return_tensors="pt"`：要求返回 PyTorch 张量。

处理完成后，`inputs` 通常包含 `input_ids`、`attention_mask` 和 `pixel_values` 等字段。其中，`pixel_values` 保存经过预处理的图片张量，`input_ids` 则表示经过分词的文本输入。

因此，最终得到的 `inputs` 是同时承载文本和视觉输入的模型输入字典。

In [ ]:
image_file = "000000039769.jpg"
raw_image = Image.open(image_file)

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(images=raw_image, text=prompt, return_tensors="pt").to(device, torch.bfloat16)

In [ ]:
print(inputs.keys())

## 调用模型进行推理

`max_new_tokens` 建议设置在 128 以上，过短会导致答案被截断。对于客观视觉问答，推荐 `temperature=0.1~0.2`、`top_p=0.9`；若希望描述更具多样性，可适当提高 `temperature`。

`batch_decode` 返回的序列包含了输入提示词。如需仅查看模型生成的回答，可执行 `generate_ids[:, inputs.input_ids.shape[1]:]` 后再解码。

In [ ]:
generate_ids = model.generate(**inputs, max_new_tokens=128)
# generate_ids[:, inputs.input_ids.shape[1]:]
processor.batch_decode(generate_ids, skip_special_tokens=True)